In [2]:
# Install PySpark if needed (mainly for Colab)
try:
    import pyspark
    print(f"PySpark already installed: version {pyspark.__version__}")
except ImportError:
    print("Installing PySpark...")
    %pip install pyspark -q
    import pyspark
    print(f"PySpark installed: version {pyspark.__version__}")

PySpark already installed: version 4.1.1


In [3]:
# Check Python version
import sys
print(f"Python version: {sys.version}")

Python version: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]


In [4]:
# Import libraries
import os
import time
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("✅ Config loaded")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("Python:", sys.executable)


✅ Config loaded
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot
HADOOP_HOME: C:\hadoop
Python: C:\Users\User\anaconda3\envs\pyspark_env\python.exe


In [6]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab1")
         .getOrCreate())

print("✅ Spark version:", spark.version)


✅ Spark version: 4.1.1


In [7]:
# Create SparkSession - the entry point to all Spark functionality
# 
# Configuration explained:
# - appName: Identifies your application in the Spark UI and logs
# - master: "local[*]" means run locally using ALL available CPU cores
#   - "local[1]" would use only 1 core (no parallelism)
#   - "local[4]" would use exactly 4 cores
#   - In production, this would be "yarn", "k8s://...", etc.
# - spark.driver.memory: Memory allocated to the driver process (default is 1g)
#   - Increase this if you're collecting large results or have many partitions

spark = SparkSession.builder \
    .appName("Lab1-NYC-Taxi") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")
print(f"Number of cores available: {spark.sparkContext.defaultParallelism}")
print("\n⚠️ Keep the Spark UI URL handy - you'll need it for exercises!")

Spark version: 4.1.1
Spark UI available at: http://127.0.0.1:4040
Number of cores available: 4

⚠️ Keep the Spark UI URL handy - you'll need it for exercises!


In [8]:
# Create data directory if it doesn't exist
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

# NYC Taxi data URL (January 2024)
data_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
data_file = os.path.join(data_dir, "yellow_tripdata_2024-01.parquet")

# Download if not already present
if not os.path.exists(data_file):
    print(f"Downloading NYC Taxi data (~50MB)...")
    print(f"URL: {data_url}")
    urllib.request.urlretrieve(data_url, data_file)
    print(f"✓ Downloaded to {data_file}")
else:
    print(f"✓ Data already exists at {data_file}")

# Check file size
file_size_mb = os.path.getsize(data_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")

URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
✓ Downloaded to data\yellow_tripdata_2024-01.parquet
File size: 47.6 MB


In [9]:
# Load the Parquet file
df = spark.read.parquet(data_file)

# Show schema
print("Dataset Schema:")
df.printSchema()

Dataset Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [10]:
# View first few rows
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [11]:
# Basic statistics
print("Dataset Statistics:")
row_count = df.count()
print(f"Total trips: {row_count:,}")

Dataset Statistics:
Total trips: 2,964,624


In [12]:
# Check the current number of partitions
num_partitions = df.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

# See how many rows are in each partition
# glom() collects each partition into a list, then we count the length
partition_sizes = df.rdd.glom().map(len).collect()
print(f"\nRows per partition: {partition_sizes}")
print(f"Min: {min(partition_sizes):,}, Max: {max(partition_sizes):,}, Avg: {sum(partition_sizes)//len(partition_sizes):,}")

Number of partitions: 4

Rows per partition: [1048576, 1048576, 0, 867472]
Min: 0, Max: 1,048,576, Avg: 741,156


In [15]:
df_repart = df.repartition(8)
df_coal = df.coalesce(2)

df_repart.rdd.getNumPartitions(), df_coal.rdd.getNumPartitions()


(8, 2)

<h2>Exercise 1 — RDD operations (sum of squares)</h2>

In [16]:
taxi_rdd = df.rdd
taxi_rdd.take(2)


[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 0, 9, 36), passenger_count=1, trip_distance=1.8, RatecodeID=1, store_and_fwd_flag='N', PULocationID=140, DOLocationID=236, payment_type=1, fare_amount=10.0, extra=3.5, mta_tax=0.5, tip_amount=3.75, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=18.75, congestion_surcharge=2.5, Airport_fee=0.0)]

In [17]:
df_from_rdd = taxi_rdd.toDF()
df_from_rdd.printSchema()


root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [18]:
from pyspark.sql import functions as F

In [21]:
rdd = spark.sparkContext.parallelize(range(1, 101))
answer_1 = rdd.map(lambda x: x*x).sum()

print("Exercise 1 - Sum of squares (1..100):", answer_1)


Exercise 1 - Sum of squares (1..100): 338350


In [23]:
answer_2 = df.count()
print("Exercise 2 - Total trips:", answer_2)


Exercise 2 - Total trips: 2964624


In [24]:
df_pass = df.filter(F.col("passenger_count") > 4)
answer_3 = df_pass.count()

print("Exercise 3 - Trips with passenger_count > 4:", answer_3)
df_pass.select("passenger_count", "PULocationID", "DOLocationID", "total_amount").show(5, truncate=False)


Exercise 3 - Trips with passenger_count > 4: 55919
+---------------+------------+------------+------------+
|passenger_count|PULocationID|DOLocationID|total_amount|
+---------------+------------+------------+------------+
|5              |239         |143         |11.62       |
|5              |143         |170         |38.16       |
|5              |246         |246         |21.32       |
|5              |229         |140         |12.9        |
|6              |148         |231         |21.36       |
+---------------+------------+------------+------------+
only showing top 5 rows


In [25]:
answer_4 = df.agg(F.sum("total_amount").alias("total_revenue")).collect()[0]["total_revenue"]
print("Exercise 4 - Total revenue:", answer_4)


Exercise 4 - Total revenue: 79456384.2800809


In [26]:
df_cc = df.filter((F.col("payment_type") == 1) & (F.col("fare_amount") > 0))

df_cc_tip = df_cc.withColumn("tip_pct", (F.col("tip_amount") / F.col("fare_amount")) * 100)

answer_5 = df_cc_tip.agg(F.avg("tip_pct").alias("avg_tip_pct")).collect()[0]["avg_tip_pct"]

print("Exercise 5 - Average tip % (credit card):", answer_5)
df_cc_tip.select("fare_amount", "tip_amount", "tip_pct").show(5, truncate=False)


Exercise 5 - Average tip % (credit card): 28.6956849921654
+-----------+----------+------------------+
|fare_amount|tip_amount|tip_pct           |
+-----------+----------+------------------+
|10.0       |3.75      |37.5              |
|23.3       |3.0       |12.875536480686694|
|10.0       |2.0       |20.0              |
|7.9        |3.2       |40.50632911392405 |
|29.6       |6.9       |23.31081081081081 |
+-----------+----------+------------------+
only showing top 5 rows


In [27]:
popular_pickups = (df.groupBy("PULocationID")
                     .count()
                     .orderBy(F.desc("count")))

top_row = popular_pickups.first()
answer_6 = int(top_row["PULocationID"])

print("Exercise 6 - Most popular PULocationID:", answer_6)
popular_pickups.show(10)


Exercise 6 - Most popular PULocationID: 132
+------------+------+
|PULocationID| count|
+------------+------+
|         132|145240|
|         161|143471|
|         237|142708|
|         236|136465|
|         162|106717|
|         230|106324|
|         186|104523|
|         142|104080|
|         138| 89533|
|         239| 88474|
+------------+------+
only showing top 10 rows


In [28]:
answer_7 = ["narrow", "wide", "narrow", "wide"]
print("Exercise 7 - Narrow vs Wide:", answer_7)


Exercise 7 - Narrow vs Wide: ['narrow', 'wide', 'narrow', 'wide']


In [29]:
(df.groupBy("PULocationID")
   .count()
   .orderBy(F.desc("count"))
   .show(20))


+------------+------+
|PULocationID| count|
+------------+------+
|         132|145240|
|         161|143471|
|         237|142708|
|         236|136465|
|         162|106717|
|         230|106324|
|         186|104523|
|         142|104080|
|         138| 89533|
|         239| 88474|
|         163| 85692|
|         170| 83064|
|          68| 77686|
|          48| 76476|
|         234| 76157|
|         141| 73946|
|         249| 66227|
|         140| 66060|
|          79| 65935|
|         164| 64912|
+------------+------+
only showing top 20 rows


In [30]:
df_small = df.select("PULocationID", "fare_amount", "tip_amount", "total_amount").filter(F.col("fare_amount") > 0)

df_small.cache()
df_small.count()  # materialize cache

print("Exercise 9 - Cached df_small and materialized it with count()")


Exercise 9 - Cached df_small and materialized it with count()


In [36]:
try:
    spark.stop()
except:
    pass


In [1]:
import os, sys
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin" + os.pathsep + os.environ.get("PATH","")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"


In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab1")
         .config("spark.hadoop.io.nativeio.NativeIO$Windows.enabled", "false")
         .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
         .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
         .getOrCreate())


In [3]:
df = spark.read.parquet(r"data\yellow_tripdata_2024-01.parquet")


In [7]:
import os
home_out = os.path.join(os.path.expanduser("~"), "spark_out", "test_parquet")
os.makedirs(os.path.dirname(home_out), exist_ok=True)

df.limit(1000).write.mode("overwrite").parquet(home_out)
print("✅ wrote to:", home_out)


Py4JJavaError: An error occurred while calling o62.parquet.
: org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/C:/Users/User/spark_out/test_parquet. SQLSTATE: 58030
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:814)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.enrichWriteError(FileFormatDataWriter.scala:92)
	at org.apache.spark.sql.execution.datasources.FileFormatDataWriter.commit(FileFormatDataWriter.scala:123)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$executeTask$1(FileFormatWriter.scala:407)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1337)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:418)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:107)
	at org.apache.spark.r

In [ ]:
jan1 = df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2024-01-01"))

out_path = "jan1_trips.parquet"
jan1.write.mode("overwrite").parquet(out_path)

answer_10 = spark.read.parquet(out_path).count()
answer_10


In [8]:
pipeline = (
    df.select(
        "PULocationID",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type"
    )
    .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0))
    .withColumn("tip_pct", (F.col("tip_amount") / F.col("fare_amount")) * 100)
)

result_11 = (
    pipeline.groupBy("PULocationID")
            .agg(
                F.count("*").alias("trip_count"),
                F.avg("tip_pct").alias("avg_tip_pct"),
                F.sum("total_amount").alias("total_revenue")
            )
            .orderBy(F.desc("trip_count"))
)

print("Exercise 11 - Analysis pipeline result (top 10 pickup locations):")
result_11.show(10, truncate=False)


Exercise 11 - Analysis pipeline result (top 10 pickup locations):
+------------+----------+------------------+--------------------+
|PULocationID|trip_count|avg_tip_pct       |total_revenue       |
+------------+----------+------------------+--------------------+
|161         |140142    |21.642882095050563|3357302.929999966   |
|237         |140121    |22.80700383406257 |2772089.649999949   |
|132         |138450    |14.185380070497319|1.1198256159997843E7|
|236         |133962    |22.568942536039806|2714062.809999956   |
|162         |104345    |21.91171190124048 |2434416.5599999763  |
|230         |102958    |20.615659819684456|2766396.2699999707  |
|186         |102153    |20.89030493626906 |2462594.4299999895  |
|142         |101794    |22.44305875284073 |2171532.0299999816  |
|138         |87697     |21.252619040334274|5820951.6999999     |
|239         |86468     |22.763166876445272|1834602.470000001   |
+------------+----------+------------------+--------------------+
only showi

In [9]:
df_small = df.select("total_amount")  # efficient: only one column read
answer_12 = df_small.filter(F.col("total_amount") > 100).count()

print("Exercise 12 - Trips with total_amount > 100:", answer_12)


Exercise 12 - Trips with total_amount > 100: 39976


In [10]:
expensive = (df.select("tpep_pickup_datetime", "PULocationID", "DOLocationID", "total_amount")
               .orderBy(F.desc("total_amount")))

print("Exercise 12 - Top 10 most expensive trips:")
expensive.show(10, truncate=False)


Exercise 12 - Top 10 most expensive trips:
+--------------------+------------+------------+------------+
|tpep_pickup_datetime|PULocationID|DOLocationID|total_amount|
+--------------------+------------+------------+------------+
|2024-01-20 11:18:47 |264         |264         |5000.0      |
|2024-01-20 11:19:33 |264         |264         |5000.0      |
|2024-01-20 11:20:15 |264         |264         |2500.0      |
|2024-01-20 11:27:48 |264         |264         |2500.0      |
|2024-01-24 13:44:43 |264         |264         |2500.0      |
|2024-01-14 10:08:11 |220         |220         |2225.3      |
|2024-01-02 07:50:08 |168         |265         |1617.5      |
|2024-01-20 11:22:27 |264         |264         |1000.0      |
|2024-01-06 21:01:38 |132         |265         |940.93      |
|2024-01-22 16:40:43 |265         |265         |900.0       |
+--------------------+------------+------------+------------+
only showing top 10 rows


In [11]:
spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion()


'3.4.2'